
# WiSig — Module 3 v3: Pseudo-Label Improvement Suite

这版 notebook 基于你现有的 Module 3 v2 思路，专门围绕 **“提高伪标签准确率”** 做一次性综合实验。

固定框架：
- Module 2 仍使用：
  - `Sid = z(EmbMaha) + 0.5 z(Energy)`
  - `Sdom = V1_mixNLL`
- 仍只对 `gate == Known-Drift` 的样本做适配
- 仍使用 embedding residual adapter

本 notebook 一次性比较以下伪标签策略：

- `NoAdapt`
- `UngatedAdapt`（负对照）
- `GatedAdapt`（v2 baseline，自硬标签）
- `GatedSelfSoft`
- `GatedProtoAgree`
- `GatedProtoFusionSoft`
- `GatedTriAgree`
- `GatedTriFusionSoft`
- `OracleGatePseudoAdapt`
- `OracleLabelAdapt`

其中：
- `Proto` = source prototype 几何判别
- `Tri` = logit + prototype + kNN vote
- `Soft` = 用软目标而不是 one-hot 硬标签
- `Agree` = 多路一致后才纳入伪标签池
- `Fusion` = 多路概率融合后作为软伪标签

重点输出：
1. 各方法在 drift 样本上的 **伪标签准确率**
2. 各方法在适配流上的 **selection quality**
3. 各方法的 **最终 drift_acc / FAR / FRR**


In [1]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)


TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4
KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 5

TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3
D_DIM = 32

# fixed Module 2
FUSION_LAM = 0.5
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# stream split
ADAPT_FRAC = 0.5
MAX_ADAPT_PER_SET = 2000
MAX_EVAL_PER_SET = 3000

# adapter
ADAPTER_BOTTLENECK = 64
ADAPTER_SCALE = 0.3
ADAPT_EPOCHS = 20
ADAPT_LR = 1e-3
ADAPT_WEIGHT_DECAY = 1e-4
ADAPT_BATCH = 128

# replay
REPLAY_PER_CLASS = 200
LAMBDA_REPLAY_CE = 1.0
LAMBDA_REPLAY_ANCHOR = 1.0

# pseudo-label suite params
PSEUDO_MIN_KEEP = 64
SOFT_T = 2.0
PROTO_T = 0.10
KNN_TOPK = 15
KNN_MEM_PER_CLASS = 300

# fusion weights
W_LOGIT = 1.0
W_PROTO = 1.0
W_KNN = 1.0

# confidence threshold from stable source-like set A
CONF_STABLE_Q = 0.10

ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module3_pseudo_suite/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM, FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    ADAPT_FRAC=ADAPT_FRAC, MAX_ADAPT_PER_SET=MAX_ADAPT_PER_SET, MAX_EVAL_PER_SET=MAX_EVAL_PER_SET,
    ADAPTER_BOTTLENECK=ADAPTER_BOTTLENECK, ADAPTER_SCALE=ADAPTER_SCALE,
    ADAPT_EPOCHS=ADAPT_EPOCHS, ADAPT_LR=ADAPT_LR, ADAPT_WEIGHT_DECAY=ADAPT_WEIGHT_DECAY, ADAPT_BATCH=ADAPT_BATCH,
    REPLAY_PER_CLASS=REPLAY_PER_CLASS, LAMBDA_REPLAY_CE=LAMBDA_REPLAY_CE, LAMBDA_REPLAY_ANCHOR=LAMBDA_REPLAY_ANCHOR,
    PSEUDO_MIN_KEEP=PSEUDO_MIN_KEEP, SOFT_T=SOFT_T, PROTO_T=PROTO_T, KNN_TOPK=KNN_TOPK, KNN_MEM_PER_CLASS=KNN_MEM_PER_CLASS,
    W_LOGIT=W_LOGIT, W_PROTO=W_PROTO, W_KNN=W_KNN, CONF_STABLE_Q=CONF_STABLE_Q,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_module3_pseudo_suite/run_20260308_212144


In [2]:
def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x):
    return (x[...,0] + 1j*x[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X, d_dim=32):
    Xc = iq_to_complex(X)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
SOURCE_RXS: [np.str_('1-1'), np.str_('7-14'), np.str_('7-7')]
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']


In [3]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [4]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((len(X_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def closed_set_acc_from_logits(logits, y_true):
    return float((np.argmax(logits,1) == y_true).mean())

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

In [5]:
def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_embmaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def sid_energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore_fixed(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

def sid_fusion_fixed(Z, logits, mu_z, var_z, ref_sid_emb_A, ref_sid_en_A, lam=0.5):
    return (zscore_fixed(sid_embmaha(Z, mu_z, var_z), ref_sid_emb_A) +
            lam * zscore_fixed(sid_energy(logits), ref_sid_en_A)).astype(np.float32)

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps = []
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum()+1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

def calibrate_module2_thresholds(Sid_A, Sdom_A, frr_target=0.05, false_drift_target=0.05):
    tau_id = float(np.quantile(Sid_A, frr_target))
    tau_dom = float(np.quantile(Sdom_A, 1.0 - false_drift_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def compute_module2_scores_from_cached(Z, logits, D, mu_z_src, var_z_src,
                                       ref_sid_emb_A, ref_sid_en_A,
                                       models_kr, fallback_k, source_rx_ids,
                                       tau_id=None, tau_dom=None):
    k_hat = np.argmax(logits, axis=1).astype(np.int64)
    Sid = sid_fusion_fixed(Z, logits, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, lam=FUSION_LAM)
    Sdom = sdom_V1_mixNLL(D, k_hat, models_kr, fallback_k, source_rx_ids)
    out = dict(Sid=Sid, Sdom=Sdom, k_hat=k_hat)
    if tau_id is not None and tau_dom is not None:
        out["gate_pred"] = gate_predict(Sid, Sdom, tau_id, tau_dom)
    return out

In [6]:
def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (ez.sum(axis=1, keepdims=True) + 1e-12)

def normalize_rows(P):
    P = np.asarray(P, dtype=np.float32)
    P = np.maximum(P, 1e-12)
    P = P / P.sum(axis=1, keepdims=True)
    return P.astype(np.float32)

def msp_from_probs(P):
    return P.max(axis=1).astype(np.float32)

def msp_from_logits(logits):
    return msp_from_probs(softmax_np(logits))

def onehot_from_labels(y, K):
    y = np.asarray(y, dtype=np.int64)
    out = np.zeros((len(y), K), dtype=np.float32)
    out[np.arange(len(y)), y] = 1.0
    return out

def pseudo_acc_from_probs(P, y_true):
    pred = np.argmax(P, axis=1)
    mask = y_true >= 0
    return float((pred[mask] == y_true[mask]).mean()) if np.any(mask) else float("nan")

def fit_class_prototypes(Z_train, y_train, K, l2norm=True):
    Z = Z_train.copy().astype(np.float32)
    if l2norm:
        Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    protos = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        z = Z[y_train == k]
        protos[k] = z.mean(axis=0)
    if l2norm:
        protos = protos / (np.linalg.norm(protos, axis=1, keepdims=True) + 1e-12)
    return protos

def proto_probs_cosine(Z, protos, temp=0.10, l2norm=True):
    Zx = Z.astype(np.float32)
    if l2norm:
        Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    scores = (Zx @ protos.T) / max(temp, 1e-6)
    return softmax_np(scores)

def build_knn_memory(Z_train, y_train, per_class=300, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_train):
        idx = np.where(y_train == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    Zm = Z_train[keep].astype(np.float32)
    ym = y_train[keep].astype(np.int64)
    Zm = Zm / (np.linalg.norm(Zm, axis=1, keepdims=True) + 1e-12)
    return Zm, ym

def knn_class_probs(Z, Zm, ym, K, topk=15):
    Zx = Z.astype(np.float32)
    Zx = Zx / (np.linalg.norm(Zx, axis=1, keepdims=True) + 1e-12)
    sims = Zx @ Zm.T
    topk = min(topk, Zm.shape[0])
    idx = np.argpartition(-sims, kth=topk-1, axis=1)[:, :topk]
    probs = np.zeros((Zx.shape[0], K), dtype=np.float32)
    for i in range(Zx.shape[0]):
        ii = idx[i]
        ss = sims[i, ii]
        ww = np.exp(ss - ss.max())
        for j, w in zip(ii, ww):
            probs[i, ym[j]] += float(w)
    return normalize_rows(probs)

def weighted_prob_fusion(prob_list, weights):
    out = np.zeros_like(prob_list[0], dtype=np.float32)
    for P, w in zip(prob_list, weights):
        out += float(w) * P.astype(np.float32)
    return normalize_rows(out)

def select_idx_by_mask_with_fallback(base_idx, mask_keep, conf, min_keep=64):
    base_idx = np.asarray(base_idx, dtype=np.int64)
    if len(base_idx) == 0:
        return base_idx
    idx_keep = base_idx[mask_keep]
    if len(idx_keep) >= min_keep:
        return idx_keep
    order = np.argsort(-conf[base_idx])
    k = min(len(base_idx), min_keep)
    return base_idx[order[:k]]

def normalize_conf_weights(conf):
    conf = np.asarray(conf, dtype=np.float32)
    if len(conf) == 0:
        return conf
    cmin, cmax = float(conf.min()), float(conf.max())
    if cmax - cmin < 1e-12:
        return np.ones_like(conf, dtype=np.float32)
    return (0.5 + 0.5 * (conf - cmin) / (cmax - cmin)).astype(np.float32)

def summarize_method_on_selected(idx_sel, y_true, P):
    if len(idx_sel) == 0:
        return dict(sel_size=0, pseudo_acc=float("nan"))
    mask = y_true[idx_sel] >= 0
    if np.any(mask):
        pa = float((np.argmax(P[idx_sel], axis=1)[mask] == y_true[idx_sel][mask]).mean())
    else:
        pa = float("nan")
    return dict(sel_size=int(len(idx_sel)), pseudo_acc=pa)

In [7]:
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list, rx_id_list = [], [], [], []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
        np.concatenate(rx_id_list,0).astype(np.int64),
    )

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
    )

def split_adapt_eval(X, y, D, frac=0.5, max_adapt=None, max_eval=None, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X))
    cut = int(len(idx) * frac)
    idx_ad = idx[:cut]
    idx_ev = idx[cut:]
    if max_adapt is not None and len(idx_ad) > max_adapt:
        idx_ad = idx_ad[:max_adapt]
    if max_eval is not None and len(idx_ev) > max_eval:
        idx_ev = idx_ev[:max_eval]
    return X[idx_ad], y[idx_ad], D[idx_ad], X[idx_ev], y[idx_ev], D[idx_ev]

def concat_sets(items):
    X = np.concatenate([it[0] for it in items], axis=0)
    y = np.concatenate([it[1] for it in items], axis=0)
    D = np.concatenate([it[2] for it in items], axis=0)
    return X, y, D

In [8]:
class EmbDatasetWeightedHard(Dataset):
    def __init__(self, Z, y, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        if w is None:
            w = np.ones((len(Z),), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.y[i], self.w[i]

class EmbDatasetWeightedSoft(Dataset):
    def __init__(self, Z, q, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.q = torch.tensor(q, dtype=torch.float32)
        if w is None:
            w = np.ones((len(Z),), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.q[i], self.w[i]

class ResidualAdapter(nn.Module):
    def __init__(self, dim=512, bottleneck=64, scale=0.3):
        super().__init__()
        self.fc1 = nn.Linear(dim, bottleneck)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(bottleneck, dim)
        self.scale = scale
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, z):
        return z + self.scale * self.fc2(self.relu(self.fc1(z)))

def apply_adapter_and_fc(adapter, fc_layer, Z_np, batch=512):
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    if adapter is None:
        with torch.no_grad():
            z = torch.tensor(Z_np, dtype=torch.float32)
            logits = fc(z).numpy().astype(np.float32)
        return Z_np.astype(np.float32), logits

    fc = fc.to(device)
    for p in fc.parameters():
        p.requires_grad = False
    adapter.eval()

    ds = EmbDatasetWeightedHard(Z_np, np.zeros((len(Z_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z2_all, logits_all = [], []
    with torch.no_grad():
        for zb,_,_ in dl:
            zb = zb.to(device)
            z2 = adapter(zb)
            logits = fc(z2)
            Z2_all.append(z2.cpu().numpy().astype(np.float32))
            logits_all.append(logits.cpu().numpy().astype(np.float32))
    return np.concatenate(Z2_all,0), np.concatenate(logits_all,0)

def make_replay_subset(Z_src, y_src, per_class=200, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_src):
        idx = np.where(y_src == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    return Z_src[keep], y_src[keep]

def adapt_on_embeddings_hard(fc_layer, Z_target, y_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None

    adapter = ResidualAdapter(dim=Z_target.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_t = EmbDatasetWeightedHard(Z_target, y_target, target_weights)
    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_t = DataLoader(ds_t, batch_size=batch, shuffle=True, drop_last=False)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=True)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce_none = nn.CrossEntropyLoss(reduction="none")
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    for ep in range(epochs):
        adapter.train()
        it_s = iter(dl_s)
        for zt, yt, wt in dl_t:
            try:
                zs, ys, _ = next(it_s)
            except StopIteration:
                it_s = iter(dl_s)
                zs, ys, _ = next(it_s)

            zt, yt, wt = zt.to(device), yt.to(device), wt.to(device)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zt2 = adapter(zt)
            zs2 = adapter(zs)
            logits_t = fc(zt2)
            logits_s = fc(zs2)

            loss_t = (ce_none(logits_t, yt) * wt).mean()
            loss_s = ce(logits_s, ys)
            loss_anchor = mse(zs2, zs)
            loss = loss_t + lambda_replay_ce * loss_s + lambda_replay_anchor * loss_anchor
            loss.backward()
            opt.step()
    return adapter

def adapt_on_embeddings_soft(fc_layer, Z_target, q_target, Z_replay, y_replay,
                             target_weights=None,
                             bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                             lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None

    adapter = ResidualAdapter(dim=Z_target.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_t = EmbDatasetWeightedSoft(Z_target, q_target, target_weights)
    ds_s = EmbDatasetWeightedHard(Z_replay, y_replay, None)
    dl_t = DataLoader(ds_t, batch_size=batch, shuffle=True, drop_last=False)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=True)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    for ep in range(epochs):
        adapter.train()
        it_s = iter(dl_s)
        for zt, qt, wt in dl_t:
            try:
                zs, ys, _ = next(it_s)
            except StopIteration:
                it_s = iter(dl_s)
                zs, ys, _ = next(it_s)

            zt, qt, wt = zt.to(device), qt.to(device), wt.to(device)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zt2 = adapter(zt)
            zs2 = adapter(zs)
            logits_t = fc(zt2)
            logits_s = fc(zs2)

            logp_t = torch.log_softmax(logits_t, dim=1)
            kl = torch.sum(qt * (torch.log(qt + 1e-12) - logp_t), dim=1)
            loss_t = (kl * wt).mean()
            loss_s = ce(logits_s, ys)
            loss_anchor = mse(zs2, zs)
            loss = loss_t + lambda_replay_ce * loss_s + lambda_replay_anchor * loss_anchor
            loss.backward()
            opt.step()
    return adapter

def plot_bar_compare(metric_dict, out_png, title):
    names = list(metric_dict.keys())
    vals = [metric_dict[k] for k in names]
    plt.figure(figsize=(10,4))
    plt.bar(range(len(names)), vals)
    plt.xticks(range(len(names)), names, rotation=25, ha="right")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

In [9]:
def build_pseudo_method_bank(K, gate_pred, y_true_adapt, P_logit_adapt, P_proto_adapt, P_knn_adapt, min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    idx_all = np.arange(len(gate_pred))

    P_tri = weighted_prob_fusion([P_logit_adapt, P_proto_adapt, P_knn_adapt], [W_LOGIT, W_PROTO, W_KNN])
    P_lp = weighted_prob_fusion([P_logit_adapt, P_proto_adapt], [W_LOGIT, W_PROTO])

    y_logit = np.argmax(P_logit_adapt, axis=1)
    y_proto = np.argmax(P_proto_adapt, axis=1)
    y_knn = np.argmax(P_knn_adapt, axis=1)
    y_tri = np.argmax(P_tri, axis=1)

    conf_logit = msp_from_probs(P_logit_adapt)
    conf_proto = msp_from_probs(P_proto_adapt)
    conf_knn = msp_from_probs(P_knn_adapt)
    conf_lp = msp_from_probs(P_lp)
    conf_tri = msp_from_probs(P_tri)

    mask_lp_agree = (y_logit == y_proto)
    mask_tri_agree = (y_logit == y_proto) & (y_proto == y_knn)

    idx_lp = select_idx_by_mask_with_fallback(idx_gate, mask_lp_agree[idx_gate], conf_lp, min_keep=min_keep)
    idx_tri = select_idx_by_mask_with_fallback(idx_gate, mask_tri_agree[idx_gate], conf_tri, min_keep=min_keep)

    idx_oracle = np.where(y_true_adapt >= 0)[0]

    methods = {
        "GatedAdapt": dict(mode="hard", idx=idx_gate, y=y_logit[idx_gate], q=None, w=np.ones((len(idx_gate),), dtype=np.float32), P=P_logit_adapt),
        "GatedSelfSoft": dict(mode="soft", idx=idx_gate, y=None, q=normalize_rows(P_logit_adapt[idx_gate] ** (1.0 / SOFT_T)), w=normalize_conf_weights(conf_logit[idx_gate]), P=P_logit_adapt),
        "GatedProtoAgree": dict(mode="hard", idx=idx_lp, y=y_logit[idx_lp], q=None, w=normalize_conf_weights(conf_lp[idx_lp]), P=onehot_from_labels(y_logit, K)),
        "GatedProtoFusionSoft": dict(mode="soft", idx=idx_gate, y=None, q=P_lp[idx_gate], w=normalize_conf_weights(conf_lp[idx_gate]), P=P_lp),
        "GatedTriAgree": dict(mode="hard", idx=idx_tri, y=y_tri[idx_tri], q=None, w=normalize_conf_weights(conf_tri[idx_tri]), P=onehot_from_labels(y_tri, K)),
        "GatedTriFusionSoft": dict(mode="soft", idx=idx_gate, y=None, q=P_tri[idx_gate], w=normalize_conf_weights(conf_tri[idx_gate]), P=P_tri),
        "UngatedAdapt": dict(mode="hard", idx=idx_all, y=y_logit[idx_all], q=None, w=np.ones((len(idx_all),), dtype=np.float32), P=P_logit_adapt),
        "OracleGatePseudoAdapt": dict(mode="hard", idx=idx_oracle, y=y_logit[idx_oracle], q=None, w=np.ones((len(idx_oracle),), dtype=np.float32), P=P_logit_adapt),
        "OracleLabelAdapt": dict(mode="hard", idx=idx_oracle, y=y_true_adapt[idx_oracle], q=None, w=np.ones((len(idx_oracle),), dtype=np.float32), P=onehot_from_labels(np.clip(y_true_adapt, 0, K-1), K)),
    }

    info = {}
    for name, spec in methods.items():
        idx = spec["idx"]
        stats = summarize_method_on_selected(idx, y_true_adapt, spec["P"])
        sel_precision = float((y_true_adapt[idx] >= 0).mean()) if len(idx) > 0 else float("nan")
        sel_recall = float(np.sum(y_true_adapt[idx] >= 0) / max(1, np.sum(y_true_adapt >= 0))) if len(idx) > 0 else 0.0
        sel_keep = float(len(idx) / max(1, len(y_true_adapt)))
        info[name] = dict(sel_precision=sel_precision, sel_recall=sel_recall, sel_keep_ratio=sel_keep,
                          sel_size=stats["sel_size"], pseudo_acc_selected=stats["pseudo_acc"], pseudo_acc=stats["pseudo_acc"])
    return methods, info, P_lp, P_tri

In [10]:
def evaluate_variant(adapter, model_fc, Z_A, y_A, D_A, Z_B, y_B, D_B, Z_C, y_C, D_C, Z_U, D_U,
                     mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                     tau_id, tau_dom):
    Z_A2, logits_A2 = apply_adapter_and_fc(adapter, model_fc, Z_A, batch=512)
    Z_B2, logits_B2 = apply_adapter_and_fc(adapter, model_fc, Z_B, batch=512)
    Z_C2, logits_C2 = apply_adapter_and_fc(adapter, model_fc, Z_C, batch=512)
    Z_U2, logits_U2 = apply_adapter_and_fc(adapter, model_fc, Z_U, batch=512)

    sc_A = compute_module2_scores_from_cached(Z_A2, logits_A2, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_B = compute_module2_scores_from_cached(Z_B2, logits_B2, D_B, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_C = compute_module2_scores_from_cached(Z_C2, logits_C2, D_C, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
    sc_U = compute_module2_scores_from_cached(Z_U2, logits_U2, D_U, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)

    stable_acc = closed_set_acc_from_logits(logits_A2, y_A)
    drift_acc_rx = closed_set_acc_from_logits(logits_B2, y_B)
    drift_acc_day = closed_set_acc_from_logits(logits_C2, y_C)
    drift_acc_all = float((np.sum(np.argmax(logits_B2,1) == y_B) + np.sum(np.argmax(logits_C2,1) == y_C)) / (len(y_B) + len(y_C)))

    FRR_stable = float(1.0 - (sc_A["gate_pred"] == 0).mean())
    FAR_unknown_accept = float((sc_U["gate_pred"] == 0).mean())
    miss_drift_rx = float((sc_B["gate_pred"] == 0).mean())
    miss_drift_day = float((sc_C["gate_pred"] == 0).mean())
    miss_drift_all = float((np.sum(sc_B["gate_pred"] == 0) + np.sum(sc_C["gate_pred"] == 0)) / (len(sc_B["gate_pred"]) + len(sc_C["gate_pred"])))
    auc_unknown_eval = roc_auc(np.concatenate([np.zeros_like(sc_A["Sid"], dtype=np.int64), np.ones_like(sc_U["Sid"], dtype=np.int64)]), np.concatenate([-sc_A["Sid"], -sc_U["Sid"]]))

    return dict(
        stable_acc=stable_acc,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        FRR_stable=FRR_stable,
        FAR_unknown_accept=FAR_unknown_accept,
        miss_drift_rx=miss_drift_rx,
        miss_drift_day=miss_drift_day,
        miss_drift_all=miss_drift_all,
        auc_unknown_eval=auc_unknown_eval,
    )

In [11]:
def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    X_B, y_B, D_B = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_C, y_C, D_C = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_D, y_D, D_D = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
    X_E, y_E, D_E = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_F, y_F, D_F = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        for ep in range(1, EPOCHS + 1):
            _ = run_epoch(model, train_loader, optimizer=opt)
            _, va_acc = run_epoch(model, val_loader, optimizer=None)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        model.load_state_dict(best_state)
        torch.save(best_state, os.path.join(fold_dir, "best_source_model.pth"))

        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        D_A = D_src[idx_te]

        mu_z_src, var_z_src = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        ref_sid_emb_A = sid_embmaha(Z_A, mu_z_src, var_z_src)
        ref_sid_en_A  = sid_energy(logits_A)
        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        models_kr, fallback_k = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)
        sc_A = compute_module2_scores_from_cached(Z_A, logits_A, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids)
        tau_id, tau_dom = calibrate_module2_thresholds(sc_A["Sid"], sc_A["Sdom"], FRR_TARGET, FALSE_DRIFT_TARGET)

        protos = fit_class_prototypes(Z_tr, y_src[idx_train], K, l2norm=True)
        Zm, ym = build_knn_memory(Z_tr, y_src[idx_train], per_class=KNN_MEM_PER_CLASS, seed=SEED + 31*fold)

        P_logit_A = softmax_np(logits_A)
        tau_conf = float(np.quantile(msp_from_probs(P_logit_A), CONF_STABLE_Q))

        B_sp = split_adapt_eval(X_B, y_B, D_B, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=10000+fold)
        C_sp = split_adapt_eval(X_C, y_C, D_C, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=11000+fold)
        D_sp = split_adapt_eval(X_D, y_D, D_D, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=12000+fold)
        E_sp = split_adapt_eval(X_E, y_E, D_E, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=13000+fold)
        F_sp = split_adapt_eval(X_F, y_F, D_F, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=14000+fold)

        X_adapt, y_adapt, D_adapt = concat_sets([
            (B_sp[0], B_sp[1], B_sp[2]),
            (C_sp[0], C_sp[1], C_sp[2]),
            (D_sp[0], D_sp[1], D_sp[2]),
            (E_sp[0], E_sp[1], E_sp[2]),
            (F_sp[0], F_sp[1], F_sp[2]),
        ])

        X_B_ev, y_B_ev, D_B_ev = B_sp[3], B_sp[4], B_sp[5]
        X_C_ev, y_C_ev, D_C_ev = C_sp[3], C_sp[4], C_sp[5]
        X_U_ev, y_U_ev, D_U_ev = concat_sets([
            (D_sp[3], D_sp[4], D_sp[5]),
            (E_sp[3], E_sp[4], E_sp[5]),
            (F_sp[3], F_sp[4], F_sp[5]),
        ])

        logits_adapt, Z_adapt = infer_logits_embed(model, X_adapt, batch=512)
        sc_adapt = compute_module2_scores_from_cached(Z_adapt, logits_adapt, D_adapt, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom)
        gate_pred_adapt = sc_adapt["gate_pred"]

        P_logit_adapt = softmax_np(logits_adapt)
        P_proto_adapt = proto_probs_cosine(Z_adapt, protos, temp=PROTO_T, l2norm=True)
        P_knn_adapt   = knn_class_probs(Z_adapt, Zm, ym, K, topk=KNN_TOPK)

        methods, method_info, P_lp_adapt, P_tri_adapt = build_pseudo_method_bank(
            K, gate_pred_adapt, y_adapt,
            P_logit_adapt, P_proto_adapt, P_knn_adapt,
            min_keep=PSEUDO_MIN_KEEP
        )

        logits_B_ev, Z_B_ev = infer_logits_embed(model, X_B_ev, batch=512)
        logits_C_ev, Z_C_ev = infer_logits_embed(model, X_C_ev, batch=512)
        logits_U_ev, Z_U_ev = infer_logits_embed(model, X_U_ev, batch=512)
        P_logit_B = softmax_np(logits_B_ev)
        P_proto_B = proto_probs_cosine(Z_B_ev, protos, temp=PROTO_T, l2norm=True)
        P_knn_B   = knn_class_probs(Z_B_ev, Zm, ym, K, topk=KNN_TOPK)
        P_lp_B    = weighted_prob_fusion([P_logit_B, P_proto_B], [W_LOGIT, W_PROTO])
        P_tri_B   = weighted_prob_fusion([P_logit_B, P_proto_B, P_knn_B], [W_LOGIT, W_PROTO, W_KNN])

        P_logit_C = softmax_np(logits_C_ev)
        P_proto_C = proto_probs_cosine(Z_C_ev, protos, temp=PROTO_T, l2norm=True)
        P_knn_C   = knn_class_probs(Z_C_ev, Zm, ym, K, topk=KNN_TOPK)
        P_lp_C    = weighted_prob_fusion([P_logit_C, P_proto_C], [W_LOGIT, W_PROTO])
        P_tri_C   = weighted_prob_fusion([P_logit_C, P_proto_C, P_knn_C], [W_LOGIT, W_PROTO, W_KNN])

        pseudo_eval = {
            "logit": dict(rx=pseudo_acc_from_probs(P_logit_B, y_B_ev), day=pseudo_acc_from_probs(P_logit_C, y_C_ev)),
            "lp":    dict(rx=pseudo_acc_from_probs(P_lp_B,    y_B_ev), day=pseudo_acc_from_probs(P_lp_C,    y_C_ev)),
            "tri":   dict(rx=pseudo_acc_from_probs(P_tri_B,   y_B_ev), day=pseudo_acc_from_probs(P_tri_C,   y_C_ev)),
        }
        pseudo_eval["logit"]["all"] = float((np.sum(np.argmax(P_logit_B,1) == y_B_ev) + np.sum(np.argmax(P_logit_C,1) == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["lp"]["all"]    = float((np.sum(np.argmax(P_lp_B,1)    == y_B_ev) + np.sum(np.argmax(P_lp_C,1)    == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))
        pseudo_eval["tri"]["all"]   = float((np.sum(np.argmax(P_tri_B,1)   == y_B_ev) + np.sum(np.argmax(P_tri_C,1)   == y_C_ev)) / (len(y_B_ev) + len(y_C_ev)))

        Z_replay, y_replay = make_replay_subset(Z_tr, y_src[idx_train], per_class=REPLAY_PER_CLASS, seed=SEED+fold)

        adapters = {"NoAdapt": None}
        for name, spec in methods.items():
            idx = spec["idx"]
            if spec["mode"] == "hard":
                adapters[name] = adapt_on_embeddings_hard(
                    model.fc, Z_adapt[idx], spec["y"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )
            else:
                adapters[name] = adapt_on_embeddings_soft(
                    model.fc, Z_adapt[idx], spec["q"], Z_replay, y_replay, target_weights=spec["w"],
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE, epochs=ADAPT_EPOCHS,
                    lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                )

        fold_metrics = dict(
            split=split_id, fold=fold,
            tau_id=tau_id, tau_dom=tau_dom, tau_conf=tau_conf,
            pseudo_logit_rx=pseudo_eval["logit"]["rx"], pseudo_logit_day=pseudo_eval["logit"]["day"], pseudo_logit_all=pseudo_eval["logit"]["all"],
            pseudo_lp_rx=pseudo_eval["lp"]["rx"],       pseudo_lp_day=pseudo_eval["lp"]["day"],       pseudo_lp_all=pseudo_eval["lp"]["all"],
            pseudo_tri_rx=pseudo_eval["tri"]["rx"],     pseudo_tri_day=pseudo_eval["tri"]["day"],     pseudo_tri_all=pseudo_eval["tri"]["all"],
        )

        for name, info in method_info.items():
            fold_metrics[f"{name}_sel_precision"] = info["sel_precision"]
            fold_metrics[f"{name}_sel_recall"] = info["sel_recall"]
            fold_metrics[f"{name}_sel_keep"] = info["sel_keep_ratio"]
            fold_metrics[f"{name}_pseudo_acc_selected"] = info["pseudo_acc_selected"]
            fold_metrics[f"{name}_sel_size"] = info["sel_size"]

        variants_order = ["NoAdapt"] + list(methods.keys())
        for name in variants_order:
            adapter = adapters[name]
            fold_metrics[name] = evaluate_variant(
                adapter, model.fc,
                Z_A, y_src[idx_te], D_A,
                Z_B_ev, y_B_ev, D_B_ev,
                Z_C_ev, y_C_ev, D_C_ev,
                Z_U_ev, D_U_ev,
                mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids,
                tau_id, tau_dom
            )

        plot_bar_compare({k: fold_metrics[k]["drift_acc_all"] for k in variants_order}, os.path.join(fold_dir, "drift_acc_all_compare.png"), "Drift accuracy (all) compare")
        plot_bar_compare({k: fold_metrics[k]["FAR_unknown_accept"] for k in variants_order}, os.path.join(fold_dir, "far_unknown_compare.png"), "Unknown FAR compare")

        with open(os.path.join(fold_dir, "metrics_module3_pseudo_suite.json"), "w", encoding="utf-8") as f:
            json.dump(fold_metrics, f, indent=2)

        rows.append(fold_metrics)
        print(
            f"[split {split_id} fold {fold}] "
            f"pLogit={pseudo_eval['logit']['all']:.3f} pLP={pseudo_eval['lp']['all']:.3f} pTri={pseudo_eval['tri']['all']:.3f} | "
            f"NoAdapt={fold_metrics['NoAdapt']['drift_acc_all']:.3f} "
            f"Gated={fold_metrics['GatedAdapt']['drift_acc_all']:.3f} "
            f"ProtoSoft={fold_metrics['GatedProtoFusionSoft']['drift_acc_all']:.3f} "
            f"TriSoft={fold_metrics['GatedTriFusionSoft']['drift_acc_all']:.3f} "
            f"OracleLbl={fold_metrics['OracleLabelAdapt']['drift_acc_all']:.3f}"
        )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for split_id in range(1, TX_SPLIT_REPEATS + 1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)


=== TX split 1 ===
KNOWN_TX: [np.str_('20-15'), np.str_('20-19'), np.str_('14-10'), np.str_('8-20')]
UNKNOWN_TX: [np.str_('6-15'), np.str_('14-7')]
[split 1 fold 1] pLogit=0.597 pLP=0.605 pTri=0.630 | NoAdapt=0.597 Gated=0.650 ProtoSoft=0.653 TriSoft=0.655 OracleLbl=0.956
[split 1 fold 2] pLogit=0.546 pLP=0.551 pTri=0.572 | NoAdapt=0.546 Gated=0.579 ProtoSoft=0.581 TriSoft=0.584 OracleLbl=0.953
[split 1 fold 3] pLogit=0.576 pLP=0.585 pTri=0.618 | NoAdapt=0.576 Gated=0.659 ProtoSoft=0.656 TriSoft=0.670 OracleLbl=0.951
[split 1 fold 4] pLogit=0.594 pLP=0.595 pTri=0.615 | NoAdapt=0.594 Gated=0.621 ProtoSoft=0.632 TriSoft=0.636 OracleLbl=0.951
[split 1 fold 5] pLogit=0.721 pLP=0.720 pTri=0.744 | NoAdapt=0.721 Gated=0.723 ProtoSoft=0.760 TriSoft=0.735 OracleLbl=0.957

=== TX split 2 ===
KNOWN_TX: [np.str_('8-20'), np.str_('6-15'), np.str_('20-15'), np.str_('14-10')]
UNKNOWN_TX: [np.str_('20-19'), np.str_('14-7')]
[split 2 fold 1] pLogit=0.606 pLP=0.618 pTri=0.642 | NoAdapt=0.606 Gated=0.63

In [12]:
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

summary = {}
for key in [
    "pseudo_logit_rx","pseudo_logit_day","pseudo_logit_all",
    "pseudo_lp_rx","pseudo_lp_day","pseudo_lp_all",
    "pseudo_tri_rx","pseudo_tri_day","pseudo_tri_all",
    "tau_id","tau_dom","tau_conf"
]:
    summary[key] = dict(zip(["mean","std"], mean_std([r[key] for r in all_rows])))

variants = [
    "NoAdapt","UngatedAdapt","GatedAdapt","GatedSelfSoft","GatedProtoAgree",
    "GatedProtoFusionSoft","GatedTriAgree","GatedTriFusionSoft",
    "OracleGatePseudoAdapt","OracleLabelAdapt"
]
metrics = ["stable_acc","drift_acc_rx","drift_acc_day","drift_acc_all","FRR_stable","FAR_unknown_accept","miss_drift_rx","miss_drift_day","miss_drift_all","auc_unknown_eval"]

summary["variants"] = {}
for v in variants:
    summary["variants"][v] = {}
    for m in metrics:
        vals = [r[v][m] for r in all_rows]
        summary["variants"][v][m] = dict(zip(["mean","std"], mean_std(vals)))

pseudo_methods = [
    "GatedAdapt","GatedSelfSoft","GatedProtoAgree","GatedProtoFusionSoft",
    "GatedTriAgree","GatedTriFusionSoft","UngatedAdapt","OracleGatePseudoAdapt","OracleLabelAdapt"
]
summary["selection"] = {}
for pm in pseudo_methods:
    summary["selection"][pm] = {}
    for sk in ["sel_precision","sel_recall","sel_keep","pseudo_acc_selected"]:
        vals = [r[f"{pm}_{sk}"] for r in all_rows]
        summary["selection"][pm][sk] = dict(zip(["mean","std"], mean_std(vals)))

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Pseudo-label accuracy on true drift pools ===")
for k in ["pseudo_logit_all","pseudo_lp_all","pseudo_tri_all"]:
    print(f"{k:24s}: {summary[k]['mean']:.3f} ± {summary[k]['std']:.3f}")

print("\n=== Selection quality on adaptation stream ===")
for pm in pseudo_methods:
    print(f"\n[{pm}]")
    for sk in ["sel_precision","sel_recall","sel_keep","pseudo_acc_selected"]:
        print(f"{sk:24s}: {summary['selection'][pm][sk]['mean']:.3f} ± {summary['selection'][pm][sk]['std']:.3f}")

for v in variants:
    print(f"\n=== {v} ===")
    for m in metrics:
        print(f"{m:24s}: {summary['variants'][v][m]['mean']:.3f} ± {summary['variants'][v][m]['std']:.3f}")

=== Pseudo-label accuracy on true drift pools ===
pseudo_logit_all        : 0.663 ± 0.059
pseudo_lp_all           : 0.666 ± 0.057
pseudo_tri_all          : 0.689 ± 0.053

=== Selection quality on adaptation stream ===

[GatedAdapt]
sel_precision           : 0.556 ± 0.300
sel_recall              : 0.148 ± 0.015
sel_keep                : 0.165 ± 0.121
pseudo_acc_selected     : 0.957 ± 0.063

[GatedSelfSoft]
sel_precision           : 0.556 ± 0.300
sel_recall              : 0.148 ± 0.015
sel_keep                : 0.165 ± 0.121
pseudo_acc_selected     : 0.957 ± 0.063

[GatedProtoAgree]
sel_precision           : 0.556 ± 0.300
sel_recall              : 0.148 ± 0.015
sel_keep                : 0.165 ± 0.121
pseudo_acc_selected     : 0.957 ± 0.063

[GatedProtoFusionSoft]
sel_precision           : 0.556 ± 0.300
sel_recall              : 0.148 ± 0.015
sel_keep                : 0.165 ± 0.121
pseudo_acc_selected     : 0.957 ± 0.063

[GatedTriAgree]
sel_precision           : 0.556 ± 0.300
sel_recall 